# 02a · Feature engineering (I): de filas cliente-vehículo a una tabla de clientes

El CSV que nos pasan no tiene una fila por cliente, tiene una fila por cada **relación cliente-vehículo**. Si un cliente ha tenido cuatro coches, aparece cuatro veces. Pero lo que nos piden es segmentar **clientes**, así que antes de hacer nada necesitamos una tabla con una fila por cliente.

Esta es la primera parte del reto 2. Aquí montamos esa tabla paso a paso, creamos las variables derivadas y justificamos cada una: de dónde sale, qué vimos en los datos y por qué la queremos. En la segunda parte (`02b_pipeline`) elegimos cuáles entran al modelo y las preparamos con un pipeline de sklearn.

La lógica está en `src/clean.py` para que los tres usemos exactamente la misma tabla; aquí la vamos llamando por partes y comprobando cada decisión. Algunas decisiones salen del EDA (`01_eda_calidad`) y lo indicamos cuando es así.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import config, clean

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## 1. Qué tenemos realmente en el CSV

Cargamos los datos. `load_raw` solo lee el fichero, quita la primera columna (un índice sin nombre que viene de la exportación) y convierte la fecha de matriculación a fecha.

In [ ]:
raw = clean.load_raw()
raw.shape

In [ ]:
print(f"Filas:     {len(raw):,}")
print(f"Clientes:  {raw.client_id.nunique():,}")
print(f"Vehículos: {raw.vehicle_id.nunique():,}")

Tenemos 302.850 filas pero solo 173.456 clientes. Y hay menos vehículos que filas, o sea que un mismo coche puede aparecer con varios clientes (por ejemplo el dueño y otra persona que lo usa).

Vamos a ver cuántos vehículos tiene cada cliente.

In [ ]:
veh_por_cliente = raw.groupby("client_id").size()
veh_por_cliente.describe()

In [ ]:
grupos = {"1 vehículo": veh_por_cliente == 1,
          "más de 1": veh_por_cliente > 1,
          "20 o más": veh_por_cliente >= 20}

resumen = pd.DataFrame({
    "clientes": {k: m.sum() for k, m in grupos.items()},
    "filas": {k: veh_por_cliente[m].sum() for k, m in grupos.items()},
})
resumen["% clientes"] = (resumen.clientes / len(veh_por_cliente) * 100).round(1)
resumen["% filas"] = (resumen.filas / len(raw) * 100).round(1)
resumen

Esto es lo importante:

- El 73 % de los clientes tiene un solo vehículo.
- Pero el 27 % que tiene varios se queda con el **58 % de las filas**.
- Y hay 908 clientes con 20 o más vehículos (claramente flotas) que son solo el 0,5 % de los clientes pero ocupan el 16 % de las filas. El más grande tiene 561.

Si metiéramos las filas tal cual en K-Means pasarían varias cosas malas:

1. Un cliente con 11 coches contaría 11 veces, así que los clusters los acabarían decidiendo las flotas y los clientes con muchos coches.
2. Las filas de un mismo cliente son casi iguales (comparten edad, país, consentimientos...), quedan pegadas entre sí y el silhouette saldría mejor de lo que es en realidad.
3. Un mismo cliente podría caer en dos clusters distintos según el coche. Y marketing no puede mandarle dos campañas a la misma persona.
4. En el Jackknife (reto avanzado 3), si quitamos filas en lugar de clientes, el resto de filas de ese cliente sigue dentro y la estabilidad saldría inflada.

Así que lo decidimos aquí: **una fila por cliente**.

Antes de seguir, miramos un caso concreto. Este cliente tiene 11 vehículos:

In [ ]:
ejemplo = "0005e6cc04ecf053522bdce2336695e8a28b3205f6ee769cd0585f6e6d084894"
cols = ["model_class", "engine_concept", "date_of_first_registration", "is_current_vehicle",
        "age_range", "months_as_mb_customer", "finance_leasing_status",
        "purchasing_power_per_inhabitant_eur"]
raw.loc[raw.client_id == ejemplo, cols]

Se ven dos cosas:

- Las columnas de la derecha (edad, antigüedad, financiación, poder adquisitivo) son iguales en las 11 filas: son datos **del cliente**. Las de la izquierda cambian: son datos **del vehículo**.
- Mirad el índice: las filas están en las posiciones 4-7, luego 67752, 135641, 203451... El fichero **no está ordenado por cliente**, viene en bloques. Por eso nunca podemos asumir que las filas de un cliente van seguidas: siempre agrupamos con `groupby("client_id")`.

## 2. ¿Qué columnas son del cliente y cuáles del vehículo?

Lo del ejemplo hay que comprobarlo con todos. Para cada columna miramos en qué porcentaje de clientes con varios vehículos toma más de un valor. Si nunca cambia, es una columna del cliente.

In [ ]:
multi = raw[raw.client_id.isin(veh_por_cliente[veh_por_cliente > 1].index)]
cambia = multi.groupby("client_id").nunique(dropna=False).gt(1).mean().mul(100).round(1)
cambia.sort_values(ascending=False).to_frame("% de clientes multi-vehículo en los que cambia")

Queda bastante limpio:

- **Del vehículo** (cambian mucho): el propio `vehicle_id`, modelo, fecha de matriculación, motor, si es actual, visitas a taller y toda la telemetría. También `role` (26 %) y `product_group` (18 %): un mismo cliente puede ser dueño de un coche y usuario de otro, o tener un turismo y una furgoneta.
- **Del cliente** (0 %): edad, género, país, antigüedad como cliente, vehículos conectados, financiación, los tres consentimientos, código postal y todo lo socioeconómico.
- `role_type` es la rara: cambia en un 1 % de los clientes. Lo vemos más abajo.

Con esto el plan es sencillo: las columnas del cliente las cogemos tal cual (la primera fila de cada uno, total son iguales) y las del vehículo hay que resumirlas.

## 3. Variables del cliente

Vamos una por una, porque no todas entran igual.

### Edad → `age_ord`

`age_range` viene en rangos de texto. Son rangos **ordenados** (de menos a más edad), así que los pasamos a un número del 0 al 5. Si los pasáramos a one-hot perderíamos ese orden y añadiríamos 6 columnas para decir lo mismo.

In [ ]:
edad = raw.groupby("client_id").age_range.first()
edad.value_counts().reindex(config.AGE_ORDER).to_frame("clientes")

Está bastante repartido: entre el 15 y el 21 % en cada tramo desde los 25 años, y muy pocos menores de 25 (un 5 %).

### País → `is_ch`

Solo hay dos países, Suiza (CH) y Austria (AT), así que basta con una columna 0/1. La mantenemos porque los mercados son distintos (precios, poder adquisitivo).

### Empresa o particular → `is_b2b`

Antes vimos que `role_type` cambia en un 1 % de los clientes. Además tenemos `role` (OWNER, USER, DRIVER...). Cruzamos las dos:

In [ ]:
pd.crosstab(raw.role, raw.role_type)

`DRIVER` es exactamente lo mismo que `B2B`: todas las filas DRIVER son B2B y viceversa. O sea que `role` y `role_type` nos cuentan lo mismo para lo que nos interesa.

Como un cliente puede tener un coche particular y otro de empresa, marcamos `is_b2b = 1` si **alguno** de sus vehículos es B2B. Salen 980 clientes empresa.

### Poder adquisitivo y población → `ppi` y `population`

Hay cinco columnas socioeconómicas del código postal. Miramos cómo se relacionan:

In [ ]:
socio = ["purchasing_power_per_inhabitant_eur", "purchasing_power_per_household_eur",
         "population", "number_of_households", "purchasing_power_millions_eur"]
corr = raw[socio].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            xticklabels=["pp/habitante", "pp/hogar", "población", "hogares", "pp total"],
            yticklabels=["pp/habitante", "pp/hogar", "población", "hogares", "pp total"])
plt.title("Correlación entre variables socioeconómicas")
plt.show()

Hay dos bloques que dicen prácticamente lo mismo:

- Poder adquisitivo por habitante y por hogar (0,97).
- Población, número de hogares y poder adquisitivo total (0,96-1,00). Tiene sentido: el total es básicamente población × poder adquisitivo.

Nos quedamos con una de cada bloque: **poder adquisitivo por habitante** (`ppi`, cuánto dinero hay en la zona por persona) y **población** (nos sirve para saber si la zona es urbana o rural). Meter las cinco sería contar dos veces lo mismo y darle más peso del que toca.

### Género: solo para describir, no para segmentar

In [ ]:
raw.groupby("client_id").gender_derived.first().value_counts(dropna=False, normalize=True).round(3)

El diccionario dice que el género está **inferido**, no lo ha declarado el cliente. Además está muy desequilibrado (65 % hombres, 18 % mujeres) y le falta al 17 %. Segmentar clientes por un dato inferido sobre género no nos parece buena idea, ni técnica ni éticamente. Lo guardamos en la tabla para describir los segmentos al final, pero **no entra en el modelo**.

### Lo que entra tal cual

- `tenure_months` (`months_as_mb_customer`): meses como cliente. Mide la fidelidad.
- `n_connected_now` (`currently_connected_vehicles`): cuántos vehículos tiene vinculados ahora.
- `finance_leasing_status`: si ha usado leasing o paga al contado. Se codificará en el pipeline.
- Los tres consentimientos (`consent_phone`, `consent_epost`, `consent_post`), pasados a 0/1. Nos dicen por qué canal se puede contactar al cliente, que para marketing es oro.

Añadimos también `n_consents` (cuántos canales tiene abiertos) para el perfilado.

### Antigüedad por defecto → `tenure_is_default`

En el EDA vimos algo raro en `months_as_mb_customer`. Lo volvemos a mirar aquí, contando clientes:

In [ ]:
antig = raw.groupby("client_id").months_as_mb_customer.first()
antig.value_counts().sort_index().loc[30:42].to_frame("clientes")

Hay unos 21.000 clientes con **exactamente 36 meses**, unas 9 veces más que en los meses de al lado. Una cosa así no pasa de forma natural; lo más probable es que sea un valor que se puso por defecto al cargar clientes antiguos en el sistema (en el EDA vimos además que sus coches son más viejos que la media).

Para estos clientes la antigüedad no mide su fidelidad real. Si no hacemos nada, K-Means podría montar un grupo de "clientes de 3 años" que en realidad no existe. Así que creamos el flag `tenure_is_default` (1 si la antigüedad es 36) para que el modelo sepa que ese 36 no es un 36 de verdad.

Pensamos también en calcular una antigüedad alternativa desde la matriculación de su coche más antiguo, pero lo descartamos: si el coche se compró de segunda mano, su fecha no dice nada de cuándo llegó el cliente a la marca.

In [ ]:
cli = clean.client_attributes(raw)
cli.head()

In [ ]:
cli[["consent_phone", "consent_epost", "consent_post"]].mean().round(3)

Ojo con esto: el **98 % de los clientes acepta email** y todos tienen al menos un canal abierto. El consentimiento de email casi no varía, así que no va a ayudar mucho a separar clientes. Lo dejamos en la tabla y decidimos en el pipeline si entra.

## 4. Preparamos los vehículos antes de resumirlos

Antes de agregar por cliente creamos unas cuantas columnas a nivel vehículo. Todo esto lo hace `prepare_vehicles`.

### Antigüedad del vehículo → `vehicle_age_years`

Es la variable derivada que sugiere el propio enunciado. Tenemos la fecha de primera matriculación:

In [ ]:
fechas = raw.date_of_first_registration
print("Desde:", fechas.min().date(), "  Hasta:", fechas.max().date(), "  Nulos:", fechas.isna().sum())

Las fechas van de 1987 a octubre de 2026, así que tomamos como referencia el **15/10/2026** (está en `config.REF_DATE`). La antigüedad es `(fecha de referencia − matriculación) / 365,25` en años.

¿Por qué nos importa? Porque un cliente con un coche de 7 años está mucho más cerca de cambiarlo que uno que lo matriculó hace 6 meses. Es la variable más directa para las **ofertas de vehículo nuevo**. Hay 4.182 filas sin fecha (1,4 %); las dejamos como nulo.

### Clase del modelo → `class_group`

`model_class` tiene 42 valores distintos. Con one-hot serían 42 columnas, muchas con un puñado de coches (hay 3 Maybach). Las agrupamos por **segmento de mercado**, que es como lo pensaría alguien de ventas:

In [ ]:
pd.DataFrame(
    [(grupo, len(clases), ", ".join(clases)) for grupo, clases in clean.CLASS_GROUPS.items()],
    columns=["grupo", "nº clases", "clases"],
)

Las clases sin valor (1 %) van a un grupo `OTRO`. Las miramos por encima y son sobre todo modelos nuevos (GLC 400, CLA 220...), casi todos eléctricos.

La electrificación la dejamos **fuera** de estos grupos a propósito: un EQE es un ejecutivo eléctrico, así que lo ponemos en EJECUTIVO y lo de eléctrico ya lo dice la columna de motor. Así no mezclamos dos ideas en una variable.

### Motor → `engine`

`engine_concept` tiene un 44 % de nulos. Lo primero que pensamos es que serían coches viejos, pero no:

In [ ]:
nulo = raw.engine_concept.isna()
pd.DataFrame({
    "% nulo según año": nulo.groupby(fechas.dt.year >= 2020).mean().rename({False: "antes de 2020", True: "2020 o después"}),
}).round(3)

Es igual de frecuente en coches antiguos que en nuevos, así que parece simplemente un hueco en el registro. Pero hay un caso en el que sí sabemos el motor aunque falte: la familia **EQ**, que es 100 % eléctrica.

In [ ]:
eq = raw[raw.model_class.isin(clean.EV_CLASSES)]
pd.crosstab(eq.model_class, eq.engine_concept.fillna("NULO"))

Confirmado: en las clases EQ el motor es `ELECTRIC` o está vacío, nunca otra cosa. Así que esos nulos los podemos rellenar con `ELECTRIC` sin inventarnos nada.

Pero podemos ir más allá. En el EDA se nos ocurrió mirar `vehicle_type`, la versión comercial exacta ("GLC 300 e 4MATIC", "C 220 d"...). El nombre de la versión suele delatar el motor: la "e" es híbrido enchufable, la "d" diésel... Así que para cada versión miramos qué motor tiene en las filas donde sí lo sabemos, y qué parte de sus filas tienen ese motor (lo llamamos **pureza**):

In [ ]:
conocidos = raw[raw.engine_concept.notna()]
por_version = (conocidos.groupby("vehicle_type").engine_concept
               .agg(filas="size",
                    motor=lambda s: s.mode()[0],
                    pureza=lambda s: s.value_counts(normalize=True).iloc[0]))

print("Versiones con motor conocido:", len(por_version))
print(f"Pureza media por versión: {por_version.pureza.mean():.1%}")
print(f"Pureza ponderada por filas: {(por_version.pureza * por_version.filas).sum() / por_version.filas.sum():.1%}")
por_version.sort_values("filas", ascending=False).query("pureza < 0.9").head(6)

En general funciona muy bien, pero ojo con la cifra: el 99 % es la media **por versión**. Si ponderamos por filas baja al 95 %, porque hay versiones con muchas filas que son ambiguas. Las más claras son la "GLE 300 d 4MATIC" y la "C 220 d": el mismo nombre ha sido combustión o mild hybrid según la generación del coche, y su motor más frecuente solo acierta la mitad de las veces.

Por eso no mapeamos todas: solo las versiones con **al menos un 90 % de pureza y al menos 5 filas con motor conocido**. Perdemos algo de cobertura, pero no metemos motores dudosos. Veamos cuánto recuperamos:

In [ ]:
mapa = clean.engine_map_from_vehicle_type(raw)
nulos_motor = raw.engine_concept.isna()
recuperados = raw.loc[nulos_motor, "vehicle_type"].map(mapa).notna()

print("Versiones que usamos para imputar:", len(mapa))
print(f"Nulos de motor recuperados: {recuperados.sum():,} de {nulos_motor.sum():,} ({recuperados.mean():.1%})")

Recuperamos casi la mitad de los nulos con un criterio que podemos explicar. El orden final para el motor de cada coche es:

1. El dato original, si está.
2. Si no, el de su versión comercial (solo versiones fiables).
3. Si tampoco, `ELECTRIC` si es de la familia EQ.
4. Lo que quede, `UNKNOWN`, sin darle ningún significado.

Con el motor ya limpio marcamos también `is_plugin_ev`: si el coche es eléctrico o híbrido enchufable (los que se cargan en un enchufe).

### Telemetría: qué columnas usamos

Hay 10 columnas de uso del coche. Nos quedamos solo con las **medias**: distancia por viaje, velocidad, consumo y horas de funcionamiento. Las demás no nos sirven:

In [ ]:
raw.speed_mode.value_counts().head(8)

Las columnas de moda (`*_mode`) son texto y a veces traen una lista entera de valores (`0.125,10.0625,30,...`), que parece un empate entre tramos de un histograma. No son usables. Las medianas van en tramos fijos (13 valores distintos) y dicen menos que la media.

In [ ]:
raw[["trip_distance_all_counter", "speed_all_counter"]].describe().T

Y los contadores (`*_all_counter`) miden cuántos datos se han recogido, no cómo se usa el coche, con valores de hasta 1.450 millones. Fuera.

Ya podemos preparar los vehículos:

In [ ]:
veh = clean.prepare_vehicles(raw)
veh[["model_class", "class_group", "engine_concept", "engine", "is_plugin_ev", "vehicle_age_years"]].head()

## 5. El vehículo principal

Algunas cosas no se pueden promediar: no tiene sentido "la clase media" de un cliente con un A-Class y un S-Class. Para esas variables elegimos un **vehículo principal** que represente al cliente:

1. Entre sus vehículos **actuales**, el matriculado más recientemente.
2. Si no tiene ninguno actual, el más reciente de todos.
3. Si aún hay empate, desempatamos por `vehicle_id` para que el resultado sea siempre el mismo.

La idea es quedarnos con el coche que hoy define la relación con la marca. De él sacamos la clase (`main_class_group`), el motor (`main_engine`), su antigüedad (`main_vehicle_age`) y el grupo de producto (este último solo para describir).

In [ ]:
principal = clean.pick_main_vehicle(veh)
principal.loc[ejemplo]

Nuestro cliente de los 11 coches no tiene ninguno actual, así que se queda con el más reciente: un GLC mild hybrid matriculado en enero de 2026.

## 6. Resúmenes de todos sus vehículos

El resto de información de vehículos la resumimos con todos sus coches (`aggregate_vehicles`):

| Variable | Cómo se calcula | Por qué |
|---|---|---|
| `n_vehicles` | número de vehículos del cliente | Tamaño de la relación; separa particulares de flotas |
| `has_current_vehicle` | 1 si tiene algún vehículo actual | Detecta clientes sin ninguna relación activa (posibles ex-clientes) |
| `fleet_age_mean` | antigüedad media de sus vehículos | Si renueva a menudo o aguanta los coches |
| `workshop_visits_mean` | media de visitas al taller por vehículo | Relación con el servicio postventa |
| `share_plugin_ev` | % de enchufables/eléctricos entre los de motor conocido | Afinidad con la electrificación |
| `has_telemetry` | 1 si tiene algún dato de uso | Separa coches conectados de no conectados |
| `is_heavy_account` | 1 si tiene 20 o más vehículos | Marca las cuentas tipo flota para describirlas (en el EDA vimos que el 92 % de sus filas figuran como B2C, así que `is_b2b` no las detecta) |
| `trip_dist_avg`, `speed_avg`, `fuel_cons_avg`, `op_hours_mean` | media entre los vehículos que tienen ese dato | Cómo usa el coche (ciudad, carretera, intensidad) |

Dos detalles de por qué lo hacemos así:

- Las visitas al taller las promediamos en vez de sumarlas. Si sumáramos, un cliente con 10 coches tendría muchas visitas solo por tener muchos coches, y eso ya lo cuenta `n_vehicles`.
- En `share_plugin_ev` dividimos solo entre los coches con motor conocido. Si dividiéramos entre todos, un nulo contaría como "no eléctrico" y no lo sabemos. Si el cliente no tiene ningún motor conocido, se queda en nulo.

In [ ]:
agg = clean.aggregate_vehicles(veh)
agg.loc[ejemplo]

Al principio también calculamos cuántos vehículos actuales tiene cada cliente, pero al compararlo con `currently_connected_vehicles` nos encontramos esto:

In [ ]:
n_actuales = veh.groupby("client_id").is_current_vehicle.sum()
print(f"Coinciden en el {(n_actuales == cli.n_connected_now).mean():.2%} de los clientes")
print("Clientes donde no coinciden:", (n_actuales != cli.n_connected_now).sum())

Son la misma variable (solo difieren 4 clientes de 173.456). Nos viene bien por dos lados: confirma que entendemos qué significa `is_current_vehicle`, y nos ahorra meter dos veces lo mismo. Nos quedamos con la columna original y solo guardamos el flag `has_current_vehicle`.

## 7. Juntamos todo

`build_client_table` hace todos los pasos de arriba seguidos y une las tres partes por `client_id`.

In [ ]:
clientes = clean.build_client_table(raw)
print("Forma:", clientes.shape)
print("Un cliente por fila:", clientes.index.is_unique and len(clientes) == raw.client_id.nunique())

173.456 clientes, uno por fila, 32 columnas. Vamos a ver qué nulos quedan:

In [ ]:
nulos = clientes.isna().mean().mul(100).round(1)
nulos[nulos > 0].sort_values(ascending=False).to_frame("% nulos")

- La **telemetría** falta en torno a la mitad de los clientes. No es un error: son clientes sin coche conectado. Por eso tenemos `has_telemetry`.
- `share_plugin_ev` falta en el 15 %: clientes sin ningún motor conocido ni recuperable. Sin el mapeo por versión habría sido el 34 %.
- El **género** lo tenemos para describir, no entra al modelo.
- El **poder adquisitivo y la población** faltan en un 9 %, clientes cuyo código postal no tenía datos socioeconómicos.
- Las antigüedades de vehículo y la clase faltan en menos de un 1,5 %.

A propósito **no imputamos nada aquí**. Esta tabla solo recoge hechos. Rellenar nulos, aplicar logaritmos, recortar outliers y escalar son decisiones que "aprenden" de los datos (la mediana, los percentiles...), así que van en el pipeline de sklearn. Así cuando en el Jackknife re-entrenemos con submuestras, esas decisiones se recalculan también, como debe ser.

In [ ]:
clientes.describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

clientes.main_class_group.value_counts().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Clase del vehículo principal")
axes[0].invert_yaxis()

clientes.main_engine.value_counts().plot.barh(ax=axes[1], color="seagreen")
axes[1].set_title("Motor del vehículo principal")
axes[1].invert_yaxis()

clientes.n_vehicles.clip(upper=10).value_counts().sort_index().plot.bar(ax=axes[2], color="indianred")
axes[2].set_title("Vehículos por cliente (10 = 10 o más)")
axes[2].set_yscale("log")

plt.tight_layout()
plt.show()

Primera foto de la cartera:

- Dos de cada tres clientes llevan un **compacto (36 %) o un medio (30 %)**: A, CLA, GLA/GLB, C, GLC... Las furgonetas son un 11 % y el lujo/deportivo un 4 %.
- En el motor manda la combustión (44 %), y eléctricos y enchufables juntos ya rondan el 21 % de los clientes. El `UNKNOWN` se queda en un 18 %; sin el mapeo por versión habría sido casi un 40 %.
- `n_vehicles` va en escala logarítmica: la inmensa mayoría tiene 1 o 2 coches, y luego hay una cola larga de flotas. En el pipeline habrá que aplicarle un logaritmo y recortar los extremos para que no se coman un cluster.

Guardamos la tabla para que la usen los demás notebooks:

In [ ]:
clean.save_clients(clientes)
print("Guardado en", config.CLIENTS_PKL.relative_to(config.ROOT))

## 8. Resumen: la tabla de clientes

Aquí todavía no decidimos qué entra al modelo, eso lo hacemos en `02b_pipeline` mirando correlaciones. Separamos entre lo que es **candidato** a entrar y lo que solo sirve para **describir** los segmentos.

| Columna | De dónde sale | Papel |
|---|---|---|
| `age_ord` | `age_range` pasado a 0-5 | Candidata |
| `is_ch` | `country_code` | Candidata |
| `is_b2b` | algún vehículo con `role_type = B2B` | Candidata |
| `tenure_months` | `months_as_mb_customer` | Candidata |
| `tenure_is_default` | **derivada**: antigüedad = 36 (valor por defecto) | Candidata |
| `n_connected_now` | `currently_connected_vehicles` | Candidata |
| `finance_leasing_status` | tal cual | Candidata |
| `ppi` | poder adquisitivo por habitante | Candidata |
| `population` | población del código postal | Candidata |
| `main_class_group` | **derivada**: clase del vehículo principal en 7 segmentos | Candidata |
| `main_engine` | **derivada**: motor del principal (original → versión → EQ → `UNKNOWN`) | Candidata |
| `main_vehicle_age` | **derivada**: antigüedad del vehículo principal | Candidata |
| `fleet_age_mean` | **derivada**: antigüedad media de sus vehículos | Candidata |
| `n_vehicles` | **derivada**: nº de vehículos | Candidata |
| `has_current_vehicle` | **derivada**: tiene algún vehículo actual | Candidata |
| `workshop_visits_mean` | **derivada**: visitas al taller por vehículo | Candidata |
| `share_plugin_ev` | **derivada**: % enchufables/eléctricos | Candidata |
| `has_telemetry` | **derivada**: tiene datos de uso | Candidata |
| `trip_dist_avg`, `speed_avg`, `fuel_cons_avg`, `op_hours_mean` | medias de telemetría | Candidatas (se prueban con y sin) |
| `consent_phone`, `consent_epost`, `consent_post`, `n_consents` | consentimientos | Describir: dicen por dónde contactar, no quién es el cliente |
| `is_heavy_account` | **derivada**: 20 o más vehículos | Describir |
| `age_range`, `gender_derived`, `country_code`, `main_model_class`, `main_product_group` | originales | Describir |

Lo que queda para `02b_pipeline`: elegir las features definitivas, imputar nulos, logaritmos en las variables con cola larga, recortar extremos, codificar categóricas y escalar.